# Gold - KPI mensal da operacao

Entrada: `case.gold.ft_chamados` e `case.gold.dim_atendente`, geradas pela fato v2.
Saida: `case.gold.kpi_mensal`.
Grao: **mes de abertura x canal x nivel de atendimento**.

Exemplo: julho/2026, atendimento inicial, Nivel 1. Reune todos os chamados dessa combinacao no mes inteiro, sem separar dias ou horas.
Nao depende da KPI horaria nem altera a Silver ou a fato. Nao calcula abandono.


## 1. Configuracao

Widgets de catalogo, schema e periodo seguem o notebook de demanda horaria. O periodo inclusivo e somente sinalizado, sem excluir chamados. Nao ha limite de cinco minutos ou mediana nesta tabela.
O fuso da sessao e informado no resumo; nao e alterado automaticamente.


In [ ]:
import json
import math
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType, TimestampType, TimestampNTZType


def parametro(nome, padrao):
    try:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)
    except NameError:
        return padrao


CATALOG = parametro("catalog", "case")
GOLD_SCHEMA = parametro("gold_schema", "gold")
PERIODO_INICIO = parametro("periodo_inicio", "2025-01-01")
PERIODO_FIM = parametro("periodo_fim", "2026-07-31")
TABELA_FATO = "ft_chamados"
TABELA_DESTINO = "kpi_mensal"
TABELA_DIMENSAO = "dim_atendente"

if date.fromisoformat(PERIODO_INICIO) > date.fromisoformat(PERIODO_FIM):
    raise ValueError("Periodo inicial posterior ao final")


def fq(tabela):
    return ".".join("`" + parte.replace("`", "``") + "`"
                    for parte in [CATALOG, GOLD_SCHEMA, tabela])


CHAVES = ["mes_referencia", "canal", "nivel_atendimento"]


## 2. Contratos e integridade das entradas

Exigimos uma linha por chamado nao nulo na fato e uma linha por atendente nao nulo na dimensao. Nao deduplicamos chaves arbitrariamente.
A fato precisa trazer custo numerico, id_atendente e as mesmas colunas de horarios e comparacao usadas na KPI horaria.
Custos negativos ou nao finitos interrompem o processamento; custos nulos permanecem ausentes. A KPI nao reinterpreta os custos ja tratados na Silver.


In [ ]:
if not spark.catalog.tableExists(fq(TABELA_FATO)):
    raise ValueError("Execute e valide Gold_ft_chamados_v2 antes desta KPI")

ft = spark.table(fq(TABELA_FATO))
COLUNAS = [
    "id_chamado", "id_atendente", "custo", "canal", "resolvido", "hora_abertura_chamado",
    "hora_inicio_atendimento", "hora_finalizacao_atendimento", "status_consistencia",
    "tempo_espera", "tempo_atendimento", "tempo_espera_original", "tempo_atendimento_original",
]
faltando = set(COLUNAS) - set(ft.columns)
if faltando:
    raise ValueError(f"Contrato da fato incompleto: {sorted(faltando)}")
ft = ft.select(*COLUNAS)
tipos = {campo.name: campo.dataType for campo in ft.schema.fields}
for coluna in ["hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento"]:
    if not isinstance(tipos[coluna], (TimestampType, TimestampNTZType)):
        raise ValueError(f"{coluna} deve ser timestamp, recebido: {tipos[coluna]}")
for coluna in ["custo", "tempo_espera", "tempo_atendimento", "tempo_espera_original", "tempo_atendimento_original"]:
    if not isinstance(tipos[coluna], NumericType):
        raise ValueError(f"{coluna} deve ser numerica")
for coluna in ["canal", "resolvido", "status_consistencia"]:
    if not isinstance(tipos[coluna], StringType):
        raise ValueError(f"{coluna} deve ser string")

entrada = ft.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_chamado").alias("ids_unicos"),
    F.count("id_chamado").alias("ids_preenchidos"),
    F.count(F.when(~F.col("resolvido").isin("sim", "nao"), 1)).alias("resolucao_invalida"),
).first().asDict()
if entrada["total"] != entrada["ids_unicos"] or entrada["total"] != entrada["ids_preenchidos"]:
    raise ValueError(f"Fato com chave nula ou duplicada: {entrada}")
if entrada["resolucao_invalida"]:
    raise ValueError("Resolucao fora de sim/nao/nulo na fato")

if not spark.catalog.tableExists(fq(TABELA_DIMENSAO)):
    raise ValueError("Dimensao de atendentes ausente; execute a fato v2")
dim = spark.table(fq(TABELA_DIMENSAO))
if not {"id_atendente", "nivel_atendimento"}.issubset(dim.columns):
    raise ValueError("dim_atendente precisa de id_atendente e nivel_atendimento")
dim = dim.select("id_atendente", "nivel_atendimento")
if ft.schema["id_atendente"].dataType != dim.schema["id_atendente"].dataType:
    raise ValueError("Tipos de id_atendente diferentes entre fato e dimensao")
if not isinstance(dim.schema["nivel_atendimento"].dataType, StringType):
    raise ValueError("nivel_atendimento deve ser string")
if dim.filter(F.col("id_atendente").isNull()).limit(1).count():
    raise ValueError("Dimensao com id_atendente nulo")
if dim.groupBy("id_atendente").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Dimensao com id_atendente repetido; join multiplicaria chamados")
custo_double = F.col("custo").cast("double")
if ft.filter(F.col("custo").isNotNull() & (
    (F.col("custo") < 0) | F.isnan(custo_double) | (F.abs(custo_double) == float("inf"))
)).limit(1).count():
    raise ValueError("Custo negativo ou nao finito na fato")


## 3. Base mensal e duracoes

Join LEFT por id_atendente, com dimensao previamente validada: todos os chamados sao preservados. Nivel vazio, nulo ou sem correspondencia recebe `nao_identificado`. Nao inferimos autosservico a partir de atendente ausente.

Os tempos seguem exatamente a regra da KPI horaria: cronologia consistente, timestamps necessarios preenchidos e nenhuma inversao entre os horarios disponiveis. Zero e valido; datas nao sao corrigidas. Conferimos os tempos da fato com os recalculos.

Abertura ausente permanece num grupo com mes nulo. Datas fora do periodo sao contadas, sem filtro. Tempos originais sao os campos ja tratados da Silver, preservados para comparacao.


In [ ]:
abertura = F.col("hora_abertura_chamado")
inicio = F.col("hora_inicio_atendimento")
fim = F.col("hora_finalizacao_atendimento")

def verdadeiro(cond):
    return F.coalesce(cond, F.lit(False))


def finito(coluna):
    valor = F.col(coluna).cast("double")
    return valor.isNotNull() & ~F.isnan(valor) & (F.abs(valor) < float("inf"))


inversao = verdadeiro(inicio < abertura) | verdadeiro(fim < inicio) | verdadeiro(fim < abertura)
consistente = verdadeiro(F.col("status_consistencia") == "CONSISTENTE") & ~inversao
espera_valida = consistente & abertura.isNotNull() & inicio.isNotNull()
atendimento_valido = consistente & inicio.isNotNull() & fim.isNotNull()

base_com_nivel = ft.join(dim, on="id_atendente", how="left")
if base_com_nivel.count() != entrada["total"]:
    raise ValueError("Join com atendentes alterou o total de chamados")

base = (
    base_com_nivel.withColumn("mes_referencia", F.trunc(F.to_date(abertura), "month"))
    .withColumn("canal", F.when(F.col("canal").isNull() | (F.trim("canal") == ""),
                                F.lit("sem canal")).otherwise(F.col("canal")))
    .withColumn("nivel_atendimento", F.when(
        F.col("nivel_atendimento").isNull() | (F.trim("nivel_atendimento") == ""),
        F.lit("nao_identificado")).otherwise(F.col("nivel_atendimento")))
    .withColumn("espera_seg", F.when(espera_valida, F.unix_timestamp(inicio) - F.unix_timestamp(abertura)))
    .withColumn("atendimento_seg", F.when(atendimento_valido, F.unix_timestamp(fim) - F.unix_timestamp(inicio)))
    .withColumn("horarios_validos", espera_valida & atendimento_valido)
    .withColumn("abertura_fora_periodo", verdadeiro(
        ~F.to_date(abertura).between(PERIODO_INICIO, PERIODO_FIM)))
)

# null-safe: inclusive a diferenca entre um tempo nulo e um tempo preenchido.
erros_tempos = base.filter(
    ~F.col("tempo_espera").eqNullSafe(F.col("espera_seg"))
    | ~F.col("tempo_atendimento").eqNullSafe(F.col("atendimento_seg"))
).limit(1).count()
if erros_tempos:
    raise ValueError("Tempos da fato divergem das regras/timestamps. Revisar a fato antes de publicar a KPI")


## 4. Indicadores e formulas

- qtd_chamados: todos os chamados do grupo.
- pct_resolvidos = 100 x qtd_resolvidos / qtd_chamados. Resolucao ausente e contada separadamente, sem rotular como nao resolvida.
- tempo_medio_espera_min = soma_espera_segundos / qtd_espera_valida / 60.
- tempo_medio_atendimento_min = soma_atendimento_segundos / qtd_atendimento_valido / 60.
- custo_total: soma dos custos conhecidos, em reais; nulo se nenhum custo for conhecido.
- custo_medio_por_chamado = custo_total / qtd_chamados: custo conhecido por chamado da operacao; NAO estima o custo dos registros ausentes.
- custo_medio_chamados_com_custo = custo_total / qtd_chamados_com_custo: media apenas dos chamados com custo conhecido.
- pct_cobertura_custo = 100 x qtd_chamados_com_custo / qtd_chamados.

Custos zero sao conhecidos e validos. Nao imputamos custos nulos como gratuitos. Sem denominador valido, medias ficam nulas. Soma de custos preserva o tipo numerico da fato (decimal na implementacao atual), sem conversao para double.

Para Power BI, usar somas e denominadores para recalcular indicadores nos filtros; nunca somar percentuais ou tirar media simples de medias. Custos totais devem ser apresentados com sua cobertura.


In [ ]:
def contar(cond):
    return F.count(F.when(cond, F.lit(1)))


def somar(coluna):
    return F.coalesce(F.sum(coluna), F.lit(0))


AGREGACOES = [
    F.count("*").alias("qtd_chamados"),
    F.count("custo").alias("qtd_chamados_com_custo"),
    contar(F.col("custo").isNull()).alias("qtd_chamados_sem_custo"),
    contar(F.col("nivel_atendimento") == "nao_identificado").alias("qtd_nivel_nao_identificado"),
    somar("custo").alias("soma_custo_conhecido"),
    contar(F.col("resolvido") == "sim").alias("qtd_resolvidos"),
    contar(F.col("resolvido").isin("sim", "nao")).alias("qtd_resolucao_informada"),
    contar(F.col("resolvido").isNull()).alias("qtd_resolucao_ausente"),
    F.count("espera_seg").alias("qtd_espera_valida"),
    F.count("atendimento_seg").alias("qtd_atendimento_valido"),
    contar(F.col("horarios_validos")).alias("qtd_horarios_validos"),
    contar(F.col("hora_abertura_chamado").isNull()).alias("qtd_sem_abertura"),
    contar(F.col("hora_inicio_atendimento").isNull()).alias("qtd_sem_inicio"),
    contar(F.col("abertura_fora_periodo")).alias("qtd_abertura_fora_periodo"),
    contar(F.col("status_consistencia").isNull() | (F.col("status_consistencia") != "CONSISTENTE"))
        .alias("qtd_cronologia_nao_consistente"),
    somar("espera_seg").alias("soma_espera_segundos"),
    somar("atendimento_seg").alias("soma_atendimento_segundos"),
]
for medida in ["espera", "atendimento"]:
    original = f"tempo_{medida}_original"
    par = F.col(f"{medida}_seg").isNotNull() & finito(original)
    AGREGACOES.extend([
        contar(par).alias(f"qtd_comparavel_{medida}_original"),
        contar(par & (F.col(f"{medida}_seg") != F.col(original))).alias(f"qtd_divergente_{medida}_original"),
        somar(F.when(par, F.col(original))).alias(f"soma_{medida}_original_comparavel_segundos"),
        somar(F.when(par, F.col(f"{medida}_seg"))).alias(f"soma_{medida}_calculada_comparavel_segundos"),
    ])

kpi = base.groupBy(*CHAVES).agg(*AGREGACOES)
for medida, denominador in [("espera", "qtd_espera_valida"),
                            ("atendimento", "qtd_atendimento_valido")]:
    kpi = kpi.withColumn(f"tempo_medio_{medida}_min", F.when(
        F.col(denominador) > 0,
        F.col(f"soma_{medida}_segundos") / F.col(denominador) / 60.0))
kpi = (
    kpi.withColumn("pct_resolvidos", F.when(F.col("qtd_chamados") > 0,
        100.0 * F.col("qtd_resolvidos") / F.col("qtd_chamados")))
    .withColumn("custo_total", F.when(F.col("qtd_chamados_com_custo") > 0,
        F.col("soma_custo_conhecido")))
    .withColumn("custo_medio_por_chamado", F.when(F.col("qtd_chamados") > 0,
        F.col("custo_total") / F.col("qtd_chamados")))
    .withColumn("custo_medio_chamados_com_custo", F.when(F.col("qtd_chamados_com_custo") > 0,
        F.col("custo_total") / F.col("qtd_chamados_com_custo")))
    .withColumn("pct_cobertura_custo", F.when(F.col("qtd_chamados") > 0,
        100.0 * F.col("qtd_chamados_com_custo") / F.col("qtd_chamados")))
)


## 5. Validacoes antes da gravacao

Verificamos chave agregada unica, contagens entre zero e o total, denominadores coerentes, percentuais entre 0 e 100 e conservacao de todos os totais aditivos.
O custo conhecido tambem e reconciliado diretamente com a fato anterior ao join; valores decimais sao comparados exatamente. Fonte vazia gera uma tabela vazia com esquema definido.
Estas verificacoes sao obrigatorias; nao sao logs opcionais. Nao ha displays nem contagens por grupo para debug.


In [ ]:
if kpi.groupBy(*CHAVES).count().filter(F.col("count") != 1).limit(1).count():
    raise ValueError("Chave agregada duplicada")

contagens = [c for c in kpi.columns if c.startswith("qtd_")]
invalidos = F.lit(False)
for c in contagens:
    invalidos = invalidos | F.col(c).isNull() | (F.col(c) < 0) | (F.col(c) > F.col("qtd_chamados"))
invalidos = invalidos | (F.col("qtd_chamados") <= 0)
invalidos = invalidos | (F.col("qtd_resolvidos") > F.col("qtd_resolucao_informada"))
invalidos = invalidos | (F.col("qtd_resolucao_informada") + F.col("qtd_resolucao_ausente") != F.col("qtd_chamados"))
invalidos = invalidos | (F.col("qtd_horarios_validos") > F.least("qtd_espera_valida", "qtd_atendimento_valido"))
for medida in ["espera", "atendimento"]:
    invalidos = invalidos | (F.col(f"qtd_divergente_{medida}_original") > F.col(f"qtd_comparavel_{medida}_original"))
for c in ["pct_resolvidos", "pct_cobertura_custo"]:
    invalidos = invalidos | (F.col(c).isNotNull() & ~F.col(c).between(0, 100))
invalidos = invalidos | (F.col("qtd_chamados_com_custo") + F.col("qtd_chamados_sem_custo") != F.col("qtd_chamados"))
invalidos = invalidos | (F.col("soma_custo_conhecido") < 0)
invalidos = invalidos | ((F.col("qtd_chamados_com_custo") == 0) & F.col("custo_total").isNotNull())
if kpi.filter(invalidos).limit(1).count():
    raise ValueError("Contagem, denominador ou percentual invalido")

esperado = base.agg(*AGREGACOES).first().asDict()
aditivas = list(esperado)
obtido = kpi.agg(*[somar(c).alias(c) for c in aditivas]).first().asDict()
for c in aditivas:
    if c == "soma_custo_conhecido":
        confere = (esperado[c] or 0) == (obtido[c] or 0)
    else:
        confere = math.isclose(float(esperado[c] or 0), float(obtido[c] or 0), rel_tol=1e-12, abs_tol=1e-6)
    if not confere:
        raise ValueError(f"Reconciliacao falhou em {c}: esperado={esperado[c]}, obtido={obtido[c]}")
if obtido["qtd_chamados"] != entrada["total"]:
    raise ValueError("Total agregado difere da fato")

custos_fato = ft.agg(somar("custo").alias("soma"), F.count("custo").alias("qtd")).first()
if custos_fato["soma"] != obtido["soma_custo_conhecido"] or custos_fato["qtd"] != obtido["qtd_chamados_com_custo"]:
    raise ValueError("Custos ou cobertura alterados entre a fato e o agregado")


## 6. Publicacao

Grava `kpi_mensal` em Delta somente depois das validacoes. `overwrite` permite reexecutar sobre a mesma fonte sem acumular registros. Nao altera a KPI horaria nem as tabelas de CSAT da Isabella.
Executar a fato v2 validada primeiro; esta KPI e independente da execucao da demanda horaria.
O resumo retorna volumes e cobertura financeira, nao uma media de medias.


In [ ]:
(
    kpi.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(fq(TABELA_DESTINO))
)
resumo = {
    "tabela": fq(TABELA_DESTINO),
    "qtd_chamados": entrada["total"],
    "qtd_espera_valida": obtido["qtd_espera_valida"],
    "qtd_atendimento_valido": obtido["qtd_atendimento_valido"],
    "qtd_sem_abertura": obtido["qtd_sem_abertura"],
    "qtd_abertura_fora_periodo": obtido["qtd_abertura_fora_periodo"],
    "qtd_chamados_com_custo": obtido["qtd_chamados_com_custo"],
    "qtd_chamados_sem_custo": obtido["qtd_chamados_sem_custo"],
    "qtd_nivel_nao_identificado": obtido["qtd_nivel_nao_identificado"],
    "custo_total_conhecido": str(obtido["soma_custo_conhecido"]),
    "fuso_sessao": spark.conf.get("spark.sql.session.timeZone"),
}
print(json.dumps(resumo, ensure_ascii=False))
try:
    dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False))
except NameError:
    pass
